# **INTEGRANTES**

**Completar con el nombre y la libreta universitaria de cada integrante**

**INTEGRANTE Nº1:**

**INTEGRANTE Nº2:**

# Ejercicio 5: Dinámica de solitones

En este ejercicio estudiamos numéricamente los solitones brillantes de la ecuación de Schrödinger no lineal adimensional

$$
i\,\frac{\partial a}{\partial t}
=
-\frac{\partial^2 a}{\partial x^2}
+\beta |a|^2a,
$$

tomando

$$
\beta=-1.
$$

Con este signo la ecuación es focalizante,

$$
i\,a_t=-a_{xx}-|a|^2a,
$$

y admite solitones brillantes. Usaremos estos perfiles como condiciones iniciales y los integraremos con el mismo método pseudoespectral + RK4 usado en el Ejercicio 4.

## Forma del solitón brillante

Para la ecuación focalizante

$$
i\,a_t=-a_{xx}-|a|^2a,
$$

una familia de solitones brillantes está dada por

$$
a(x,t)
=
\sqrt{2\mu}\,
\operatorname{sech}
\left[
\sqrt{\mu}\left(x-x_0-2\kappa t\right)
\right]
\exp\left(i\left[\kappa x-\left(\kappa^2-\mu\right)t\right]\right),
$$

donde $\mu>0$ controla la amplitud y el ancho, mientras que $\kappa$ controla la velocidad. En particular,

$$
v=2\kappa.
$$

En el dominio periódico $[0,2\pi)$ el solitón de la recta no es estrictamente periódico. Para evitar un salto artificial en los bordes, usamos la distancia periódica al centro $x_0$ al construir la envolvente inicial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.grid": True,
    "font.size": 11,
})

# Dominio periódico.
N = 128
L = 2*np.pi
x = L*np.arange(N)/N
dx = L/N
k = np.fft.fftfreq(N, d=1/N)

# Regla 2/3 para reducir aliasing en el término cúbico.
k_dealias = N//3


def sech(z):
    return 1/np.cosh(z)


def distancia_periodica(x, x0):
    '''Distancia firmada desde x0, envuelta al intervalo [-L/2, L/2).'''
    return (x - x0 + L/2) % L - L/2


def dealias(f_hat):
    '''Filtra modos altos para evitar aliasing en |a|^2 a.'''
    f_hat = f_hat.copy()
    f_hat[np.abs(k) >= k_dealias] = 0.0
    return f_hat


def energia(a):
    '''Aproxima E = integral |a|^2 dx.'''
    return dx*np.sum(np.abs(a)**2)


def rhs_nls(a, beta=-1):
    '''
    Lado derecho de a_t = i a_xx - i beta |a|^2 a.
    En este ejercicio beta=-1.
    '''
    a_hat = np.fft.fft(a)
    a_xx = np.fft.ifft(-(k**2)*a_hat)

    nonlinear = np.abs(a)**2*a
    nonlinear_hat = dealias(np.fft.fft(nonlinear))
    nonlinear_dealias = np.fft.ifft(nonlinear_hat)

    return 1j*a_xx - 1j*beta*nonlinear_dealias


def paso_rk4(a, dt, beta=-1):
    '''Un paso de Runge--Kutta clásico de orden 4.'''
    q1 = rhs_nls(a, beta)
    q2 = rhs_nls(a + 0.5*dt*q1, beta)
    q3 = rhs_nls(a + 0.5*dt*q2, beta)
    q4 = rhs_nls(a + dt*q3, beta)
    return a + dt*(q1 + 2*q2 + 2*q3 + q4)/6


def integrar(a0, tf, dt, beta=-1, guardar_cada=20):
    '''Integra y guarda muestras de a(t,x) y de E(t).'''
    n_steps = int(np.ceil(tf/dt))
    dt = tf/n_steps

    a = a0.copy()
    tiempos = [0.0]
    muestras = [a.copy()]
    energias = [energia(a)]

    for n in range(1, n_steps + 1):
        a = paso_rk4(a, dt, beta)
        if n % guardar_cada == 0 or n == n_steps:
            tiempos.append(n*dt)
            muestras.append(a.copy())
            energias.append(energia(a))

    return np.array(tiempos), np.array(muestras), np.array(energias)


def soliton_brillante_inicial(x0=np.pi, kappa=1.0, mu=1.0, fase=0.0):
    '''
    Solitón brillante en t=0.

    La envolvente usa distancia periódica, y la fase portadora es exp(i kappa x).
    '''
    s = distancia_periodica(x, x0)
    envolvente = np.sqrt(2*mu)*sech(np.sqrt(mu)*s)
    fase_portadora = np.exp(1j*(kappa*x + fase))
    return envolvente*fase_portadora

## 5(a): Un solitón con distintas velocidades

Tomamos un solitón centrado en $x=\pi$ y comparamos las velocidades asociadas a

$$
\kappa=1,
\qquad
\kappa=4.
$$

Como $v=2\kappa$, el segundo caso se traslada mucho más rápido. En ambos casos miramos la evolución de $|a(x,t)|$, ya que el módulo muestra directamente la envolvente brillante del solitón.

In [ ]:
tf_single = 6.0
dt = 5e-4
dt_mitad = dt/2
guardar_cada = 40

kappas = [1, 4]
resultados_single = {}
resultados_single_dt_mitad = {}

for kappa in kappas:
    a0 = soliton_brillante_inicial(x0=np.pi, kappa=kappa, mu=1.0)

    t, A, E = integrar(a0, tf=tf_single, dt=dt, guardar_cada=guardar_cada)
    resultados_single[kappa] = {"t": t, "A": A, "E": E}

    # Para dt/2 guardamos menos seguido: interesa comparar energía, no duplicar figuras.
    t2, A2, E2 = integrar(a0, tf=tf_single, dt=dt_mitad, guardar_cada=2*guardar_cada)
    resultados_single_dt_mitad[kappa] = {"t": t2, "A": A2, "E": E2}


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

for ax, kappa in zip(axes, kappas):
    t = resultados_single[kappa]["t"]
    A = resultados_single[kappa]["A"]
    im = ax.imshow(
        np.abs(A),
        origin="lower",
        aspect="auto",
        extent=[0, 2*np.pi, t[0], t[-1]],
        cmap="viridis",
    )
    ax.set_title(fr"Solitón brillante, $\kappa={kappa}$")
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$t$")
    fig.colorbar(im, ax=ax, label=r"$|a(x,t)|$")

plt.show()

### Animación de los solitones individuales

El gráfico anterior muestra la trayectoria completa en el plano $(x,t)$. Para complementar esa lectura, animamos $|a(x,t)|$ y comparamos las dos velocidades.

Como $v=2\kappa$, el caso $\kappa=4$ debe atravesar el dominio más rápido que el caso $\kappa=1$.

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML


def animar_solitones_individuales(resultados, kappas=(1, 4), paso=3):
    '''Anima |a(x,t)| para comparar solitones con distintas velocidades.'''
    fig, axes = plt.subplots(1, len(kappas), figsize=(12, 4), constrained_layout=True)
    if len(kappas) == 1:
        axes = [axes]

    lineas = []
    for ax, kappa in zip(axes, kappas):
        t = resultados[kappa]["t"]
        A = resultados[kappa]["A"]
        linea, = ax.plot(x, np.abs(A[0]), lw=2)
        ax.set_xlim(0, 2*np.pi)
        ax.set_ylim(0, 1.15*np.max(np.abs(A)))
        ax.set_xlabel(r"$x$")
        ax.set_ylabel(r"$|a(x,t)|$")
        ax.set_title(fr"$\kappa={kappa}$, $t={t[0]:.2f}$")
        lineas.append((linea, ax, t, A, kappa))

    frames = range(0, len(resultados[kappas[0]]["t"]), paso)

    def actualizar(frame):
        artistas = []
        for linea, ax, t, A, kappa in lineas:
            linea.set_ydata(np.abs(A[frame]))
            ax.set_title(fr"$\kappa={kappa}$, $t={t[frame]:.2f}$")
            artistas.append(linea)
        return artistas

    anim = animation.FuncAnimation(
        fig,
        actualizar,
        frames=frames,
        interval=60,
        blit=False,
    )
    plt.close(fig)
    return anim


anim_solitones = animar_solitones_individuales(resultados_single, kappas=(1, 4), paso=3)
HTML(anim_solitones.to_jshtml())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)

for kappa in kappas:
    t = resultados_single[kappa]["t"]
    E = resultados_single[kappa]["E"]
    ax.semilogy(t, np.abs(E - E[0]) + 1e-18, label=fr"$\kappa={kappa}$, $\Delta t$")

    t2 = resultados_single_dt_mitad[kappa]["t"]
    E2 = resultados_single_dt_mitad[kappa]["E"]
    ax.semilogy(t2, np.abs(E2 - E2[0]) + 1e-18, "--", label=fr"$\kappa={kappa}$, $\Delta t/2$")

ax.set_title("Conservación de energía para un solitón")
ax.set_xlabel(r"$t$")
ax.set_ylabel(r"$|\Delta E(t)|$")
ax.legend(ncol=2)
plt.show()

La comparación con $\Delta t/2$ permite separar un efecto físico de un error numérico: si al reducir el paso temporal mejora la conservación de $E$, entonces parte de la deriva observada venía de la discretización temporal. Para RK4 esperamos una mejora fuerte al bajar el paso, aunque en la práctica también influyen la discretización espacial, el de-aliasing y el hecho de representar un solitón de recta en un dominio periódico finito.

## 5(b): Encuentro de dos solitones

Ahora construimos dos solitones:

$$
x_1=\frac{\pi}{2},
\qquad
x_2=\frac{3\pi}{2}.
$$

Al primero le damos $\kappa=1$ y al segundo $\kappa=-1$. Como $v=2\kappa$, se mueven en sentidos opuestos y se encuentran repetidas veces por la periodicidad del dominio.

La condición inicial usada es una superposición de dos solitones. Esta superposición no es una solución exacta de dos solitones, pero sirve como dato inicial para estudiar numéricamente la interacción.

In [ ]:
def condicion_dos_solitones(kappa=1.0, mu=1.0):
    '''Dos solitones que viajan en sentidos opuestos.'''
    sol_izq = soliton_brillante_inicial(x0=np.pi/2, kappa=+kappa, mu=mu, fase=0.0)
    sol_der = soliton_brillante_inicial(x0=3*np.pi/2, kappa=-kappa, mu=mu, fase=0.0)
    return sol_izq + sol_der


tf_choque = 5.0
a0_choque = condicion_dos_solitones(kappa=1.0, mu=1.0)
t_choque, A_choque, E_choque = integrar(
    a0_choque,
    tf=tf_choque,
    dt=dt,
    guardar_cada=guardar_cada,
)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

im = axes[0].imshow(
    np.abs(A_choque),
    origin="lower",
    aspect="auto",
    extent=[0, 2*np.pi, t_choque[0], t_choque[-1]],
    cmap="viridis",
)
axes[0].set_title("Encuentro de dos solitones")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$t$")
fig.colorbar(im, ax=axes[0], label=r"$|a(x,t)|$")

axes[1].semilogy(t_choque, np.abs(E_choque - E_choque[0]) + 1e-18)
axes[1].set_title("Conservación de energía durante los encuentros")
axes[1].set_xlabel(r"$t$")
axes[1].set_ylabel(r"$|\Delta E(t)|$")

plt.show()

### Animación del encuentro

La animación del choque muestra que las dos estructuras localizadas se acercan, se superponen durante un intervalo corto y luego se separan. Esta visualización es útil para interpretar la pregunta “¿son interactuantes?”: durante el encuentro no vemos simplemente dos curvas rígidas independientes, sino una deformación transitoria de la amplitud total.

In [ ]:
def animar_choque(t, A, paso=2):
    '''Anima |a(x,t)| para el choque de dos solitones.'''
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    linea, = ax.plot(x, np.abs(A[0]), lw=2)
    ax.set_xlim(0, 2*np.pi)
    ax.set_ylim(0, 1.15*np.max(np.abs(A)))
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$|a(x,t)|$")
    ax.set_title(fr"Choque de solitones, $t={t[0]:.2f}$")

    frames = range(0, len(t), paso)

    def actualizar(frame):
        linea.set_ydata(np.abs(A[frame]))
        ax.set_title(fr"Choque de solitones, $t={t[frame]:.2f}$")
        return (linea,)

    anim = animation.FuncAnimation(
        fig,
        actualizar,
        frames=frames,
        interval=60,
        blit=False,
    )
    plt.close(fig)
    return anim


anim_choque = animar_choque(t_choque, A_choque, paso=2)
HTML(anim_choque.to_jshtml())

## 5(c): Espectro espacial antes, durante y después del choque

Para mirar el choque en Fourier graficamos

$$
|a_k(t)|^2.
$$

Comparamos tres instantes: antes del choque, cerca del primer encuentro y después. Además agregamos como referencia el espectro de un único solitón. Un solitón aislado que solo se traslada no cambia su espectro de energía, porque un desplazamiento espacial solo multiplica los coeficientes de Fourier por un factor de fase.

In [ ]:
def espectro_espacial(a):
    '''Devuelve k ordenado y |a_k|^2 con normalización 1/N.'''
    a_hat = np.fft.fft(a)/N
    return np.fft.fftshift(k), np.fft.fftshift(np.abs(a_hat)**2)


def indice_tiempo_mas_cercano(t, valor):
    return int(np.argmin(np.abs(t - valor)))


# Primer encuentro aproximado: distancia inicial pi y velocidad relativa 4.
t_colision = np.pi/4
tiempos_mirar = {
    "antes": 0.0,
    "durante": t_colision,
    "después": 2*t_colision,
}

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

for etiqueta, tiempo in tiempos_mirar.items():
    idx = indice_tiempo_mas_cercano(t_choque, tiempo)
    kk, spec = espectro_espacial(A_choque[idx])
    ax.semilogy(kk, spec + 1e-18, label=fr"{etiqueta}, $t\simeq {t_choque[idx]:.2f}$")

# Referencia: espectro de un único solitón.
kk, spec_unico = espectro_espacial(soliton_brillante_inicial(x0=np.pi, kappa=1.0, mu=1.0))
ax.semilogy(kk, spec_unico + 1e-18, "k--", lw=1.8, label="un solitón aislado")

ax.set_xlim(-25, 25)
ax.set_title("Espectro espacial antes, durante y después del choque")
ax.set_xlabel(r"$k$")
ax.set_ylabel(r"$|a_k|^2$")
ax.legend()
plt.show()

## Conclusión del ejercicio

Los solitones brillantes se mantienen localizados durante la integración: la envolvente $|a(x,t)|$ conserva una forma compacta y se traslada con velocidad compatible con $v=2\kappa$. El caso $\kappa=4$ exige más cuidado numérico porque el solitón contiene oscilaciones de fase más rápidas y recorre el dominio con mayor velocidad.

Al comparar $\Delta t$ con $\Delta t/2$, se observa que la conservación de energía mejora al reducir el paso temporal. Esto es consistente con el hecho de que RK4 tiene error temporal de orden alto, pero no conserva exactamente los invariantes de la ecuación continua.

En el choque de dos solitones, las estructuras localizadas atraviesan la zona de encuentro y luego vuelven a separarse. Hay interacción durante el encuentro: en el espacio físico se observa una deformación transitoria y en Fourier aparecen diferencias respecto del espectro de un solitón aislado. Sin embargo, la energía total permanece aproximadamente conservada, lo cual indica que el intercambio entre modos es principalmente una redistribución interna y no una pérdida numérica significativa.